# 4. Etapa 2 – Analiza exploratorie în Python

## A. Incarcarea si curatarea tabelului evaluari

In [1]:
""" 1. Încarcă angajati_curat.csv, departamente.csv și evaluari.csv. Coloanele de
dată le citești direct ca dată: pd.read_csv('angajati_curat.csv',
parse_dates=['data_nasterii', 'data_angajarii', 'data_plecarii']).
Afișează pentru fiecare tabel: shape, info(), head()."""

import pandas as pd
import matplotlib.pyplot as plt

angajati = pd.read_csv('angajati_curat.csv', parse_dates=['data_nasterii', 'data_angajarii', 'data_plecarii'])
departamente = pd.read_csv('departamente.csv')
evaluari = pd.read_csv('evaluari.csv')

for df in [angajati, departamente, evaluari]:
    print(df.shape)
    print(df.info())
    print(df.head())

(480, 21)
<class 'pandas.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id_angajat            480 non-null    int64         
 1   nume                  480 non-null    str           
 2   prenume               480 non-null    str           
 3   gen                   480 non-null    str           
 4   data_nasterii         480 non-null    datetime64[us]
 5   data_angajarii        480 non-null    datetime64[us]
 6   id_departament        480 non-null    int64         
 7   functie               480 non-null    str           
 8   nivel                 480 non-null    str           
 9   oras                  480 non-null    str           
 10  tip_contract          480 non-null    str           
 11   salariu_brut_lunar   480 non-null    str           
 12  status                480 non-null    str           
 13  data_plecarii        

###### Incarcarea fisierelor CSV
###### In acest pas am incarcat cele trei fisiere: `angajati_curat.csv`, `departamente.csv` si `evaluari.csv`.  Pentru tabelul angajati, coloanele de data sunt convertite direct in format datetime folosind `parse_dates`, pentru a prelucra ulterior datele corect.  
###### Pentru fiecare tabel am afisat: `shape` (numarul de randuri si coloane), `info()` (tipurile de date si numarul de valori nenule), `head()` (primele 5 randuri), pentru a verifica incarcarea corecta si structura datelor.




In [ ]:
"""2. Verifică în evaluari: duplicatele (duplicated()), valorile lipsă (isnull().sum()) și
valorile imposibile: scor_performanta în afara intervalului 1–5, satisfactie_angajat
în afara intervalului 1–10, ore_suplimentare_lunare peste 100."""

# Duplicate
duplicate = evaluari.duplicated().sum()
print("Numar duplicate:", duplicate)

# Valori lipsa pe coloane
missing_values = evaluari.isnull().sum()
print("\nValori lipsa pe coloane:\n", missing_values)

# Valori imposibile
# Scor_performanta in afara intervalului 1–5
nr_invalid_scor = ((evaluari['scor_performanta'] < 1) |
                   (evaluari['scor_performanta'] > 5)).sum()
print("\nScor performanta invalid:", nr_invalid_scor)

# Satisfactie_angajat in afara intervalului 1–10
nr_invalid_satisfactie = ((evaluari['satisfactie_angajat'] < 1) |
                          (evaluari['satisfactie_angajat'] > 10)).sum()
print("Satisfactie angajat invalida:", nr_invalid_satisfactie)

# Ore_suplimentare_lunare peste 100
nr_invalid_ore = (evaluari['ore_suplimentare_lunare'] > 100).sum()
print("Ore suplimentare lunare peste 100:", nr_invalid_ore)


Numar duplicate: 8

Valori lipsa pe coloane:
 id_evaluare                 0
id_angajat                  0
an_evaluare                 0
scor_performanta           30
satisfactie_angajat        25
ore_suplimentare_lunare     0
ore_training_anual          0
zile_concediu_medical       0
procent_bonus               0
dtype: int64

Scor performanta invalid: 4
Satisfactie angajat invalida: 3
Ore suplimentare lunare peste 100: 3


###### Rezumat verificari evaluari
###### In acest pas am verificat: cate duplicate exista, cate valori lipsa are fiecare coloana,cate valori imposibile exista pentru: scor_performanta (trebuie sa fie intre 1 si 5), satisfactie_angajat (trebuie sa fie intre 1 si 10), ore_suplimentare_lunare (nu trebuie sa depaseasca 100). Scopul acestei verificari este de a identifica erorile din date, pentru a le corecta ulterior.



In [4]:
"""3. Elimină duplicatele și transformă valorile imposibile în valori lipsă (NaN). Apoi decide ce faci
cu toate valorile lipsă: cele care existau deja și cele create acum. Fie elimini rândurile, fie le
completezi cu mediana coloanei. Argumentează alegerea într-o celulă Markdown."""

import numpy as np

# Eliminarea duplicatelor
evaluari = evaluari.drop_duplicates()

# Transformarea valorilor imposibile in NaN
# scor_performanta trebuie sa fie intre 1 si 5
evaluari.loc[
    (evaluari['scor_performanta'] < 1) |
    (evaluari['scor_performanta'] > 5),
    'scor_performanta'
] = np.nan

# satisfactie_angajat trebuie sa fie intre 1 si 10
evaluari.loc[
    (evaluari['satisfactie_angajat'] < 1) |
    (evaluari['satisfactie_angajat'] > 10),
    'satisfactie_angajat'
] = np.nan

# ore_suplimentare_lunare trebuie sa fie <= 100
evaluari.loc[
    evaluari['ore_suplimentare_lunare'] > 100,
    'ore_suplimentare_lunare'
] = np.nan

# Tratarea valorilor lipsa: completare cu mediana coloanei
evaluari = evaluari.fillna(evaluari.median(numeric_only=True))


###### Tratarea valorilor lipsa
###### In acest pas am eliminat duplicatele din tabel, am transformat valorile imposibile in valori lipsa (NaN) si am decis cum tratez valorile lipsa: cele existente deja si cele create acum.
###### Pentru analiza bazei de date evaluari, valorile lipsa pot afecta calculele de medii si scoruri.  Astfel am ales sa completez valorile lipsa cu **mediana coloanei**, deoarece: mediana nu este influentata de valori extreme, pastreaza distributia naturala a datelor si evita pierderea de informatii prin eliminarea randurilor.



In [5]:
"""4. Verifică dacă există evaluări pentru id_angajat care nu apar în tabelul angajati (isin()).
Elimină-le și notează câte au fost; altfel importul în SQL nu va funcționa, din cauza cheii
străine."""

# Id-urile valide din tabelul angajati
id_valide = angajati['id_angajat']

# Randuri din evaluari cu id_angajat invalid
evaluari_invalid = evaluari[~evaluari['id_angajat'].isin(id_valide)]

# Numarul de evaluari invalide
nr_invalid = len(evaluari_invalid)
print("Numar evaluari cu id_angajat invalid:", nr_invalid)

# Eliminarea lor din tabelul evaluari
evaluari = evaluari[evaluari['id_angajat'].isin(id_valide)]


Numar evaluari cu id_angajat invalid: 3


###### Verificarea cheii straine (id_angajat)
###### In acest pas am verificat daca exista evaluari pentru id_angajat care nu apare in tabelul angajati. Aceste randuri trebuie eliminate, altfel importul in SQL va esua din cauza cheii straine. Am folosit `isin()` pentru a identifica id-urile valide, astfel am identificat si eliminat 3 evaluari cu id_angajat invalid.


In [6]:
"""5. Salvează rezultatul în evaluari_curat.csv (to_csv(index=False))."""

evaluari.to_csv('evaluari_curat.csv', index=False)
